# Task 08 - Feature Set V2 (Bước 2)

**Bước 2** trong kế hoạch cải tiến model:
1. Feature importance (notebook 07 - đã xong, làm mốc tham chiếu).
2. **Thêm feature mới V2** (notebook này).
3. Thử các cách xử lý imbalance (under/over-sampling).
4. Finetune lại model mới.

Mục tiêu:
- Lấy thêm các feature **V2** đã quy hoạch sẵn ở task_05 từ bảng nguồn `clean_30d_label`
  (join theo `fullVisitorId + visit_id`, cùng grain với bảng V1):
  - Categorical chi tiết: `traffic_source_clean_model`, `traffic_medium_clean_model`, `browser_family_model`, `os_family_model`.
  - Sparse flags: `is_direct_traffic`, `is_paid_traffic`, `is_organic_traffic`, `is_referral_traffic`,
    `has_traffic_keyword`, `has_traffic_ad_content`, `has_geo_region`, `has_geo_metro`,
    `has_custom_dimension`, `is_socially_engaged`.
- Train **cùng baseline params** với 2 bộ feature: **V1** vs **V1+V2**, so PR-AUC / F1 (classifier) và RMSE-log (regressor).
- Đo permutation importance riêng cho nhóm V2 để biết feature nào thực sự đóng góp.
- Lưu bảng feature mở rộng (train+test) cho Bước 3/4 dùng lại.

> Nguyên tắc: rare-category mapping, category levels, scale_pos_weight, threshold đều **fit train-only**.
> V2 đều là thuộc tính của session hiện tại (device/traffic/geo) - được phép dùng, không phải tín hiệu purchase/revenue.

## 0. Setup & load metadata V1

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR if (NB_DIR / "data_pyspark_parquet").exists() else NB_DIR.parent
if not (PROJECT_ROOT / "data_pyspark_parquet").exists():
    PROJECT_ROOT = Path(r"g:/ds")

PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"
TRAIN_FEATURE_PATH = PARQUET_DIR / "train_user_session_features_30d"
TEST_FEATURE_PATH = PARQUET_DIR / "test_user_session_features_30d"
TRAIN_CLEAN_PATH = PARQUET_DIR / "train_sessions_clean_30d_label"
TEST_CLEAN_PATH = PARQUET_DIR / "test_sessions_clean_30d_label"
MODELS_DIR = PROJECT_ROOT / "models"

OUTPUT_DIR = NB_DIR / "feature_v2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (MODELS_DIR / "lgbm_modeling_preprocessing_30d.json").open(encoding="utf-8") as f:
    meta = json.load(f)

V1_FEATURE_COLUMNS = meta["feature_columns"]
V1_NUMERIC = meta["numeric_features"]
V1_CATEGORICAL = meta["categorical_features"]
V1_BINARY = meta["binary_features"]
V1_CATEGORY_LEVELS = meta["category_levels_fit_from_train_split"]
CLASSIFICATION_LABEL = meta["classification_label"]
REVENUE_LABEL = meta["revenue_label"]
LOG_REVENUE_LABEL = meta["log_revenue_label"]
SPLIT_CONFIG = meta["train_validation_split_config"]

KEY_COLUMNS = ["fullVisitorId", "visit_id"]

print("V1 feature count:", len(V1_FEATURE_COLUMNS))
print("Validation window:", SPLIT_CONFIG["validation_start_inclusive"], "->", SPLIT_CONFIG["validation_end_inclusive"])
print("Output dir:", OUTPUT_DIR)

V1 feature count: 27
Validation window: 2018-03-01 -> 2018-03-31
Output dir: G:\ds\week_4\feature_v2_outputs


## 1. Định nghĩa feature V2 và nguồn chunk

Bảng nguồn `clean_30d_label` lưu theo **chunk cột**. Các cột V2 nằm ở:
- chunk_003: `is_paid_traffic`, `is_organic_traffic`, `is_direct_traffic`, `is_referral_traffic`, `has_traffic_keyword`
- chunk_004: `traffic_source_clean_model`, `traffic_medium_clean_model`, `has_traffic_ad_content`, `has_geo_region`, `has_geo_metro`, `has_custom_dimension`, `is_socially_engaged`
- chunk_005: `browser_family_model`, `os_family_model`

In [2]:
V2_CATEGORICAL = [
    "traffic_source_clean_model",
    "traffic_medium_clean_model",
    "browser_family_model",
    "os_family_model",
]
V2_FLAGS = [
    "is_direct_traffic",
    "is_paid_traffic",
    "is_organic_traffic",
    "is_referral_traffic",
    "has_traffic_keyword",
    "has_traffic_ad_content",
    "has_geo_region",
    "has_geo_metro",
    "has_custom_dimension",
    "is_socially_engaged",
]
V2_FEATURES = V2_CATEGORICAL + V2_FLAGS

# chunk_id -> các cột V2 lấy từ chunk đó
V2_CHUNK_COLUMNS = {
    "chunk_003": ["is_paid_traffic", "is_organic_traffic", "is_direct_traffic", "is_referral_traffic", "has_traffic_keyword"],
    "chunk_004": ["traffic_source_clean_model", "traffic_medium_clean_model", "has_traffic_ad_content", "has_geo_region", "has_geo_metro", "has_custom_dimension", "is_socially_engaged"],
    "chunk_005": ["browser_family_model", "os_family_model"],
}

print("V2 categorical:", V2_CATEGORICAL)
print("V2 flags:", V2_FLAGS)
print("Total V2 features:", len(V2_FEATURES))

V2 categorical: ['traffic_source_clean_model', 'traffic_medium_clean_model', 'browser_family_model', 'os_family_model']
V2 flags: ['is_direct_traffic', 'is_paid_traffic', 'is_organic_traffic', 'is_referral_traffic', 'has_traffic_keyword', 'has_traffic_ad_content', 'has_geo_region', 'has_geo_metro', 'has_custom_dimension', 'is_socially_engaged']
Total V2 features: 14


## 2. Load bảng V1 + join các cột V2 từ chunk nguồn

In [3]:
def load_v1_table(path, name):
    cols = list(dict.fromkeys(
        KEY_COLUMNS + ["session_date", "has_full_30d_label_window"]
        + V1_FEATURE_COLUMNS + [CLASSIFICATION_LABEL, REVENUE_LABEL]
    ))
    pdf = pd.read_parquet(path, columns=cols, engine="pyarrow")
    pdf = pdf[pdf["has_full_30d_label_window"] == 1].copy()
    print(f"{name} V1 rows (full window):", len(pdf))
    return pdf


def load_v2_columns(clean_root, name):
    merged = None
    for chunk_id, chunk_cols in V2_CHUNK_COLUMNS.items():
        chunk_path = clean_root / chunk_id
        part = pd.read_parquet(chunk_path, columns=KEY_COLUMNS + chunk_cols, engine="pyarrow")
        merged = part if merged is None else merged.merge(part, on=KEY_COLUMNS, how="inner")
    print(f"{name} V2 rows:", len(merged), "| cols:", len(merged.columns) - len(KEY_COLUMNS))
    return merged


def build_extended_table(feature_path, clean_root, name):
    v1 = load_v1_table(feature_path, name)
    v2 = load_v2_columns(clean_root, name)
    merged = v1.merge(v2, on=KEY_COLUMNS, how="left")
    # Kiểm tra join không nhân dòng và không mất dòng
    assert len(merged) == len(v1), f"{name}: join changed row count {len(v1)} -> {len(merged)}"
    null_after = merged[V2_FEATURES].isna().sum()
    print(f"{name} extended rows:", len(merged))
    if null_after.sum() > 0:
        print(f"{name} V2 null counts after join:\n", null_after[null_after > 0])
    return merged


train_ext = build_extended_table(TRAIN_FEATURE_PATH, TRAIN_CLEAN_PATH, "train")
test_ext = build_extended_table(TEST_FEATURE_PATH, TEST_CLEAN_PATH, "test")

train V1 rows (full window): 1624078
train V2 rows: 1706613 | cols: 14
train extended rows: 1624078
test V1 rows (full window): 330036
test V2 rows: 401112 | cols: 14
test extended rows: 330036


## 3. Rare-category bucketing cho categorical V2 (fit train-only)

`traffic_source_clean_model` / `traffic_medium_clean_model` có thể high-cardinality. Ta gom các category
hiếm (xuất hiện ít) thành `__other__`, **ngưỡng học từ train**. `browser_family_model` / `os_family_model`
thường gọn nên giữ nguyên level từ train.

In [4]:
RARE_MIN_COUNT = 200          # category xuất hiện < ngưỡng -> gom vào __other__
RARE_BUCKET = "__other__"
HIGH_CARD_CATEGORICAL = ["traffic_source_clean_model", "traffic_medium_clean_model"]


def fit_rare_keep_set(train_pdf, column, min_count):
    counts = train_pdf[column].astype(str).value_counts()
    keep = counts[counts >= min_count].index.tolist()
    return set(keep)


def apply_rare(pdf, column, keep_set):
    s = pdf[column].astype(str)
    return s.where(s.isin(keep_set), RARE_BUCKET)


rare_keep_sets = {}
for col in HIGH_CARD_CATEGORICAL:
    keep = fit_rare_keep_set(train_ext, col, RARE_MIN_COUNT)
    rare_keep_sets[col] = keep
    train_ext[col] = apply_rare(train_ext, col, keep)
    test_ext[col] = apply_rare(test_ext, col, keep)
    print(f"{col}: giữ {len(keep)} category (+__other__)")

# Fit category levels cho toàn bộ V2 categorical từ train (sau rare bucketing)
V2_CATEGORY_LEVELS = {}
for col in V2_CATEGORICAL:
    levels = sorted(train_ext[col].dropna().astype(str).unique().tolist())
    if RARE_BUCKET not in levels and col in HIGH_CARD_CATEGORICAL:
        levels.append(RARE_BUCKET)
    V2_CATEGORY_LEVELS[col] = levels
    print(f"{col}: {len(levels)} levels")

traffic_source_clean_model: giữ 42 category (+__other__)
traffic_medium_clean_model: giữ 6 category (+__other__)
traffic_source_clean_model: 43 levels
traffic_medium_clean_model: 7 levels
browser_family_model: 6 levels
os_family_model: 6 levels


## 4. Time split + chuẩn bị dtype cho V1 và V1+V2

In [5]:
ALL_CATEGORICAL = V1_CATEGORICAL + V2_CATEGORICAL
ALL_BINARY = list(dict.fromkeys(V1_BINARY + V2_FLAGS))
ALL_CATEGORY_LEVELS = {**V1_CATEGORY_LEVELS, **V2_CATEGORY_LEVELS}

FEATURES_V1 = V1_FEATURE_COLUMNS
FEATURES_V1V2 = V1_FEATURE_COLUMNS + V2_FEATURES


def apply_dtypes(pdf):
    out = pdf.copy()
    numeric_cols = [c for c in FEATURES_V1V2 if c not in ALL_CATEGORICAL and c not in ALL_BINARY]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
    for col in ALL_BINARY:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0).astype("int8")
    for col in ALL_CATEGORICAL:
        out[col] = pd.Categorical(out[col].astype(str), categories=ALL_CATEGORY_LEVELS[col])
    out[CLASSIFICATION_LABEL] = pd.to_numeric(out[CLASSIFICATION_LABEL], errors="raise").astype("int8")
    out[REVENUE_LABEL] = pd.to_numeric(out[REVENUE_LABEL], errors="raise").astype("float64")
    out[LOG_REVENUE_LABEL] = np.log1p(out[REVENUE_LABEL])
    return out


def time_split(pdf):
    d = pd.to_datetime(pdf["session_date"]).dt.date
    start = pd.to_datetime(SPLIT_CONFIG["validation_start_inclusive"]).date()
    end = pd.to_datetime(SPLIT_CONFIG["validation_end_inclusive"]).date()
    train_part = pdf[d < start].copy()
    valid_part = pdf[(d >= start) & (d <= end)].copy()
    return train_part, valid_part


train_lgbm = apply_dtypes(train_ext)
train_split_pdf, validation_pdf = time_split(train_lgbm)

print("Train split rows:", len(train_split_pdf), "| Validation rows:", len(validation_pdf))
print("Train positive rate:", round(train_split_pdf[CLASSIFICATION_LABEL].mean() * 100, 3), "%")

Train split rows: 1530080 | Validation rows: 93998
Train positive rate: 1.317 %


## 5. Train classifier: so sánh V1 vs V1+V2 (cùng baseline params)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, confusion_matrix

CLASSIFIER_SEED = 42
FIXED_THRESHOLD = 0.5          # threshold cố định để so V1 vs V2 đúng kiểu (và khớp NB06)

y_train = train_split_pdf[CLASSIFICATION_LABEL]
y_valid = validation_pdf[CLASSIFICATION_LABEL]
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

BASE_CLASSIFIER_PARAMS = {
    "objective": "binary",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "scale_pos_weight": scale_pos_weight,
    "random_state": CLASSIFIER_SEED,
    "n_jobs": -1,
    "verbosity": -1,
}


def metrics_at_threshold(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, pred).tolist(),
    }


def train_eval_classifier(feature_cols):
    cat = [c for c in ALL_CATEGORICAL if c in feature_cols]
    X_tr, X_va = train_split_pdf[feature_cols], validation_pdf[feature_cols]
    model = LGBMClassifier(**BASE_CLASSIFIER_PARAMS)
    model.fit(X_tr, y_train, eval_set=[(X_va, y_valid)], eval_metric="binary_logloss", categorical_feature=cat)
    proba = model.predict_proba(X_va)[:, 1]
    # threshold tối ưu F1 (tham khảo)
    grid = np.linspace(0.05, 0.95, 19)
    f1s = [f1_score(y_valid, (proba >= t).astype(int), zero_division=0) for t in grid]
    best_t = float(grid[int(np.argmax(f1s))])
    metrics = {
        "feature_count": len(feature_cols),
        "pr_auc": float(average_precision_score(y_valid, proba)),   # không phụ thuộc threshold
        "at_fixed_threshold": metrics_at_threshold(y_valid, proba, FIXED_THRESHOLD),
        "at_f1_optimal_threshold": metrics_at_threshold(y_valid, proba, best_t),
    }
    return model, proba, metrics


clf_v1, proba_v1, cls_metrics_v1 = train_eval_classifier(FEATURES_V1)
clf_v1v2, proba_v1v2, cls_metrics_v1v2 = train_eval_classifier(FEATURES_V1V2)


def flat_row(m):
    fx, opt = m["at_fixed_threshold"], m["at_f1_optimal_threshold"]
    return {
        "pr_auc": round(m["pr_auc"], 4),
        f"recall@{FIXED_THRESHOLD}": round(fx["recall"], 4),
        f"precision@{FIXED_THRESHOLD}": round(fx["precision"], 4),
        f"f1@{FIXED_THRESHOLD}": round(fx["f1"], 4),
        "recall@f1opt": round(opt["recall"], 4),
        "precision@f1opt": round(opt["precision"], 4),
        "f1opt_threshold": opt["threshold"],
    }


classifier_comparison = pd.DataFrame({"V1": flat_row(cls_metrics_v1), "V1+V2": flat_row(cls_metrics_v1v2)}).T
print(f"So sánh ở CÙNG threshold = {FIXED_THRESHOLD} (cột @{FIXED_THRESHOLD}) + tham khảo F1-optimal:")
classifier_comparison

In [ ]:
fx1, fx2 = cls_metrics_v1["at_fixed_threshold"], cls_metrics_v1v2["at_fixed_threshold"]
print(f"=== So sánh V1 vs V1+V2 ở CÙNG threshold = {FIXED_THRESHOLD} ===")
print(f"Recall   : {fx1['recall']:.4f} -> {fx2['recall']:.4f}  (delta {fx2['recall']-fx1['recall']:+.4f})")
print(f"Precision: {fx1['precision']:.4f} -> {fx2['precision']:.4f}  (delta {fx2['precision']-fx1['precision']:+.4f})")
print(f"F1       : {fx1['f1']:.4f} -> {fx2['f1']:.4f}  (delta {fx2['f1']-fx1['f1']:+.4f})")

delta_pr_auc = cls_metrics_v1v2["pr_auc"] - cls_metrics_v1["pr_auc"]
print(f"\nPR-AUC (threshold-independent): {cls_metrics_v1['pr_auc']:.4f} -> {cls_metrics_v1v2['pr_auc']:.4f}  (delta {delta_pr_auc:+.4f})")
print("=> V2 GIÚP classifier" if delta_pr_auc > 0 else "=> V2 KHÔNG cải thiện classifier (PR-AUC)")

## 6. Train regressor (conditional revenue): so sánh V1 vs V1+V2

In [8]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

REGRESSOR_SEED = 42
BASE_REGRESSOR_PARAMS = {
    "objective": "regression",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": REGRESSOR_SEED,
    "n_jobs": -1,
    "verbosity": -1,
}

train_pos = train_split_pdf[train_split_pdf[REVENUE_LABEL] > 0]
valid_pos = validation_pdf[validation_pdf[REVENUE_LABEL] > 0]
y_train_reg = train_pos[LOG_REVENUE_LABEL]
y_valid_reg = valid_pos[LOG_REVENUE_LABEL]
y_valid_reg_rev = valid_pos[REVENUE_LABEL]
print("Train positive rows:", len(train_pos), "| Validation positive rows:", len(valid_pos))


def train_eval_regressor(feature_cols):
    cat = [c for c in ALL_CATEGORICAL if c in feature_cols]
    X_tr, X_va = train_pos[feature_cols], valid_pos[feature_cols]
    model = LGBMRegressor(**BASE_REGRESSOR_PARAMS)
    model.fit(X_tr, y_train_reg, eval_set=[(X_va, y_valid_reg)], eval_metric="rmse", categorical_feature=cat)
    pred_log = model.predict(X_va)
    pred_rev = np.expm1(pred_log)
    return model, {
        "feature_count": len(feature_cols),
        "rmse_log": float(np.sqrt(mean_squared_error(y_valid_reg, pred_log))),
        "mae_log": float(mean_absolute_error(y_valid_reg, pred_log)),
        "rmse_revenue": float(np.sqrt(mean_squared_error(y_valid_reg_rev, pred_rev))),
        "mae_revenue": float(mean_absolute_error(y_valid_reg_rev, pred_rev)),
    }


reg_v1, reg_metrics_v1 = train_eval_regressor(FEATURES_V1)
reg_v1v2, reg_metrics_v1v2 = train_eval_regressor(FEATURES_V1V2)

regressor_comparison = pd.DataFrame({"V1": reg_metrics_v1, "V1+V2": reg_metrics_v1v2}).T
print("Delta RMSE-log (V1+V2 - V1):", round(reg_metrics_v1v2["rmse_log"] - reg_metrics_v1["rmse_log"], 4),
      "(âm = tốt hơn)")
regressor_comparison

Train positive rows: 20109 | Validation positive rows: 1084
Delta RMSE-log (V1+V2 - V1): -0.0214 (âm = tốt hơn)


,feature_count,rmse_log,mae_log,rmse_revenue,mae_revenue
V1,27.0,1.075001,0.834430,388.779713,138.300676
V1+V2,41.0,1.053636,0.821074,386.539427,138.165736


## 7. Permutation importance riêng cho nhóm V2 (classifier V1+V2)

In [9]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    clf_v1v2, validation_pdf[FEATURES_V1V2], y_valid,
    scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1,
)
perm_df = pd.DataFrame({
    "feature": FEATURES_V1V2,
    "perm_mean": perm.importances_mean,
    "perm_std": perm.importances_std,
    "is_v2": [c in V2_FEATURES for c in FEATURES_V1V2],
}).sort_values("perm_mean", ascending=False).reset_index(drop=True)

print("Permutation importance - nhóm V2:")
v2_perm = perm_df[perm_df["is_v2"]].copy()
print(v2_perm.to_string(index=False))

# V2 feature có đóng góp dương
useful_v2 = v2_perm.loc[v2_perm["perm_mean"] > 0, "feature"].tolist()
weak_v2 = v2_perm.loc[v2_perm["perm_mean"] <= 0, "feature"].tolist()
print("\nV2 hữu ích (perm > 0):", useful_v2)
print("V2 yếu (perm <= 0):", weak_v2)

Permutation importance - nhóm V2:
                   feature  perm_mean  perm_std  is_v2
traffic_source_clean_model   0.025814  0.005097   True
      browser_family_model   0.007201  0.002454   True
           os_family_model   0.005665  0.001141   True
         is_direct_traffic   0.001604  0.000308   True
       is_referral_traffic   0.001135  0.000154   True
traffic_medium_clean_model   0.001037  0.000251   True
            has_geo_region   0.000721  0.000178   True
             has_geo_metro   0.000638  0.000185   True
    has_traffic_ad_content   0.000592  0.000509   True
      has_custom_dimension   0.000581  0.000313   True
       has_traffic_keyword   0.000068  0.000221   True
           is_paid_traffic   0.000003  0.000035   True
       is_socially_engaged   0.000000  0.000000   True
        is_organic_traffic  -0.000137  0.000075   True

V2 hữu ích (perm > 0): ['traffic_source_clean_model', 'browser_family_model', 'os_family_model', 'is_direct_traffic', 'is_referral_traffic',

## 8. Lưu kết quả so sánh + bảng feature mở rộng cho Bước 3/4

In [10]:
comparison_result = {
    "classifier": {"V1": cls_metrics_v1, "V1+V2": cls_metrics_v1v2,
                   "delta_pr_auc": delta_pr_auc, "fixed_threshold": FIXED_THRESHOLD},
    "regressor": {"V1": reg_metrics_v1, "V1+V2": reg_metrics_v1v2},
    "v2_features": V2_FEATURES,
    "v2_useful_perm_gt_0": useful_v2,
    "v2_weak_perm_le_0": weak_v2,
    "rare_min_count": RARE_MIN_COUNT,
    "high_card_categorical": HIGH_CARD_CATEGORICAL,
}
with (OUTPUT_DIR / "v1_vs_v1v2_comparison.json").open("w", encoding="utf-8") as f:
    json.dump(comparison_result, f, ensure_ascii=False, indent=2, default=str)

perm_df.to_csv(OUTPUT_DIR / "v1v2_permutation_importance.csv", index=False)

# Lưu metadata feature V2 (để Bước 3/4 dựng lại đúng dtype)
v2_feature_metadata = {
    "v1_feature_columns": V1_FEATURE_COLUMNS,
    "v2_features": V2_FEATURES,
    "v2_categorical": V2_CATEGORICAL,
    "v2_flags": V2_FLAGS,
    "all_categorical": ALL_CATEGORICAL,
    "all_binary": ALL_BINARY,
    "all_category_levels": ALL_CATEGORY_LEVELS,
    "rare_keep_sets": {k: sorted(v) for k, v in rare_keep_sets.items()},
    "split_config": SPLIT_CONFIG,
}
with (MODELS_DIR / "feature_v2_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(v2_feature_metadata, f, ensure_ascii=False, indent=2, default=str)

# Ghi bảng feature mở rộng (train+test) ra parquet để tái sử dụng
SAVE_COLUMNS = KEY_COLUMNS + ["session_date"] + FEATURES_V1V2 + [CLASSIFICATION_LABEL, REVENUE_LABEL]
TRAIN_V2_OUT = PARQUET_DIR / "train_user_session_features_30d_v2.parquet"
TEST_V2_OUT = PARQUET_DIR / "test_user_session_features_30d_v2.parquet"
apply_dtypes(train_ext)[SAVE_COLUMNS].to_parquet(TRAIN_V2_OUT, index=False)
apply_dtypes(test_ext)[SAVE_COLUMNS].to_parquet(TEST_V2_OUT, index=False)

print("Saved:")
print(" -", OUTPUT_DIR / "v1_vs_v1v2_comparison.json")
print(" -", MODELS_DIR / "feature_v2_metadata.json")
print(" -", TRAIN_V2_OUT)
print(" -", TEST_V2_OUT)

Saved:
 - G:\ds\week_4\feature_v2_outputs\v1_vs_v1v2_comparison.json
 - G:\ds\models\feature_v2_metadata.json
 - G:\ds\data_pyspark_parquet\train_user_session_features_30d_v2.parquet
 - G:\ds\data_pyspark_parquet\test_user_session_features_30d_v2.parquet


## 9. Nhận xét & bước tiếp theo

Cách đọc kết quả:
- So **PR-AUC** của classifier V1+V2 với V1 (mục 5). Tăng = V2 có ích cho bài toán phân loại mua hàng.
- So **RMSE-log** của regressor (mục 6). Giảm = V2 có ích cho dự đoán doanh thu.
- `v2_useful_perm_gt_0` (mục 7) cho biết feature V2 nào thực sự đóng góp -> giữ lại; nhóm `weak` cân nhắc bỏ.

Output dùng cho bước sau:
- `train/test_user_session_features_30d_v2.parquet`: bảng feature V1+V2 đã chuẩn dtype.
- `models/feature_v2_metadata.json`: feature list + category levels + rare mapping (để transform nhất quán).

**Bước 3** sẽ dùng chính bảng V1+V2 (giữ các feature đã chọn) để thử under-sampling / over-sampling,
so với baseline `scale_pos_weight`. **Bước 4** finetune lại trên bộ feature + chiến lược imbalance đã chốt.